# Propensity Score Matching — Right Heart Catheterization

Replicates the Connors et al. (1996) analysis of whether right heart catheterization (RHC)
affects 30-day survival in critically ill ICU patients.

| | |
|---|---|
| **Treatment** | `received_rhc` — received RHC on day 1 of ICU admission (0/1) |
| **Outcome** | `died_30day` — died within 30 days of admission (0/1) |
| **Sample** | 5,735 critically ill patients |
| **Source** | Connors et al., JAMA 1996; data via hbiostat.org |

**The famous result:** RHC was standard clinical practice but PSM showed it was associated with
*increased* mortality — a finding that changed ICU care worldwide.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

In [ ]:
DATA_DIR = Path("../data/raw")

df = pd.read_csv(DATA_DIR / "rhc.csv", index_col=0)

df = df.rename(columns={
    # --- Treatment & Outcome ---
    "swang1":    "received_rhc",                    # treatment: RHC on day 1 (RHC / No RHC)
    "death":     "died_180day",                     # death at any time up to 180 days (Yes/No)
    "dth30":     "died_30day",                      # death within 30 days — primary outcome (Yes/No)

    # --- Dates ---
    "sadmdte":   "study_admission_date",            # date patient entered the study
    "dschdte":   "hospital_discharge_date",         # date of hospital discharge
    "dthdte":    "date_of_death",                   # date of death (if applicable)
    "lstctdte":  "last_contact_date",               # date of last known contact
    "ptid":      "patient_id",                      # unique patient identifier

    # --- Demographics ---
    "age":       "age_years",                       # age at admission
    "sex":       "sex",                             # sex (Male / Female)
    "race":      "race",                            # race (white / black / other)
    "edu":       "education_years",                 # years of formal education
    "income":    "income",                          # income bracket
    "ninsclas":  "insurance_class",                 # insurance type (Medicare/Medicaid/Private etc.)

    # --- Disease categories ---
    "cat1":      "primary_disease_category",        # primary diagnosis category on admission
    "cat2":      "secondary_disease_category",      # secondary diagnosis category on admission
    "ca":        "cancer_status",                   # cancer diagnosis (No / Yes / Metastatic)

    # --- Admission reason indicators (binary) ---
    "resp":      "admission_respiratory",           # respiratory disease as admission reason
    "card":      "admission_cardiac",               # cardiovascular disease as admission reason
    "neuro":     "admission_neurological",          # neurological disease as admission reason
    "gastr":     "admission_gastrointestinal",      # GI disease as admission reason
    "renal":     "admission_renal",                 # renal disease as admission reason
    "meta":      "admission_metabolic",             # metabolic disease as admission reason
    "hema":      "admission_hematological",         # hematological disease as admission reason
    "seps":      "admission_sepsis",                # sepsis as admission reason
    "trauma":    "admission_trauma",                # trauma as admission reason
    "ortho":     "admission_orthopedic",            # orthopedic condition as admission reason

    # --- Medical history / comorbidities (binary 0/1) ---
    "cardiohx":  "hx_cardiac_disease",             # history of cardiac disease
    "chfhx":     "hx_congestive_heart_failure",    # history of congestive heart failure
    "dementhx":  "hx_dementia",                    # history of dementia
    "psychhx":   "hx_psychiatric_disorder",        # history of psychiatric disorder
    "chrpulhx":  "hx_chronic_pulmonary_disease",   # history of chronic pulmonary disease
    "renalhx":   "hx_renal_disease",               # history of renal disease
    "liverhx":   "hx_liver_disease",               # history of liver disease
    "gibledhx":  "hx_gi_bleeding",                 # history of GI bleeding
    "malighx":   "hx_malignancy",                  # history of malignancy
    "immunhx":   "hx_immunosuppression",           # history of immunosuppression
    "transhx":   "hx_prior_transplant",            # history of prior transplant
    "amihx":     "hx_acute_myocardial_infarction", # history of acute MI

    # --- Functional status ---
    "adld3p":    "activities_daily_living_prior",  # ADL score 3 days before admission
    "das2d3pc":  "duke_activity_status_index",     # DASI score 2 days before admission
    "dnr1":      "dnr_status_day1",                # do-not-resuscitate order on day 1 (Yes/No)

    # --- Severity scores ---
    "aps1":      "apache_score_day1",              # APACHE III severity score on day 1
    "scoma1":    "glasgow_coma_score_day1",        # Glasgow Coma Scale score on day 1
    "surv2md1":  "support_2month_survival_prob",   # SUPPORT model predicted 2-month survival
    "t3d30":     "support_survival_time_30day",    # SUPPORT model predicted survival time (30-day)

    # --- Vital signs on day 1 ---
    "meanbp1":   "mean_blood_pressure_day1",       # mean arterial blood pressure (mmHg)
    "hrt1":      "heart_rate_day1",                # heart rate (beats per minute)
    "resp1":     "respiratory_rate_day1",          # respiratory rate (breaths per minute)
    "temp1":     "temperature_celsius_day1",       # body temperature (Celsius)
    "wtkilo1":   "weight_kg_day1",                 # body weight (kg)

    # --- Lab values on day 1 ---
    "pafi1":     "pao2_fio2_ratio_day1",           # PaO2/FiO2 ratio — measure of oxygenation
    "paco21":    "paco2_day1",                     # arterial CO2 partial pressure (mmHg)
    "ph1":       "blood_ph_day1",                  # arterial blood pH
    "wblc1":     "white_blood_cell_count_day1",    # WBC count (thousands per mm³)
    "hema1":     "hematocrit_day1",                # hematocrit (%)
    "sod1":      "sodium_day1",                    # serum sodium (mEq/L)
    "pot1":      "potassium_day1",                 # serum potassium (mEq/L)
    "crea1":     "creatinine_day1",                # serum creatinine (mg/dL)
    "bili1":     "bilirubin_day1",                 # serum bilirubin (mg/dL)
    "alb1":      "albumin_day1",                   # serum albumin (g/dL)
    "urin1":     "urine_output_day1",              # urine output (mL)
})

# Convert all Yes/No string columns to binary integers
for col in ["received_rhc", "died_30day", "died_180day"]:
    df[col] = (df[col].isin(["RHC", "Yes"])).astype(int)

print(f"Shape: {df.shape}")
print(f"Received RHC  : {df.received_rhc.sum():,}")
print(f"No RHC        : {(df.received_rhc == 0).sum():,}")
print(f"\nDied (30 day) : {df.died_30day.sum():,}  ({df.died_30day.mean():.1%})")
print(f"Died (180 day): {df.died_180day.sum():,} ({df.died_180day.mean():.1%})")

In [7]:
df.head()

,cat1,cat2,ca,sadmdte,dschdte,dthdte,lstctdte,died_30day,cardiohx,chfhx,...,meta,hema,seps,trauma,ortho,adld3p,urin1,race,income,ptid
1,COPD,NaN,Yes,11142,11151.0,NaN,11382,0,0,0,...,No,No,No,No,No,0.0,NaN,white,Under $11k,5
2,MOSF w/Sepsis,NaN,No,11799,11844.0,11844.0,11844,1,1,1,...,No,No,Yes,No,No,NaN,1437.0,white,Under $11k,7
3,MOSF w/Malignancy,MOSF w/Sepsis,Yes,12083,12143.0,NaN,12400,0,0,0,...,No,No,No,No,No,NaN,599.0,white,$25-$50k,9
4,ARF,NaN,No,11146,11183.0,11183.0,11182,1,0,0,...,No,No,No,No,No,NaN,NaN,white,$11-$25k,10
5,MOSF w/Sepsis,NaN,No,12035,12037.0,12037.0,12036,1,0,0,...,No,No,No,No,No,NaN,64.0,white,Under $11k,11


In [8]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
sadmdte,5735.0,11638.686312,513.967751,10754.000000,11163.500000,11759.000000,12097.000000,12441.000000
dschdte,5734.0,11660.050401,513.447322,10757.000000,11184.000000,11777.000000,12120.000000,12560.000000
dthdte,3722.0,11753.869156,538.812330,10757.000000,11267.000000,11831.500000,12208.000000,12783.000000
lstctdte,5735.0,11781.257890,524.094168,10756.000000,11316.000000,11868.000000,12244.000000,12644.000000
died_30day,5735.0,0.648997,0.477325,0.000000,0.000000,1.000000,1.000000,1.000000
cardiohx,5735.0,0.176635,0.381393,0.000000,0.000000,0.000000,0.000000,1.000000
chfhx,5735.0,0.178030,0.382571,0.000000,0.000000,0.000000,0.000000,1.000000
dementhx,5735.0,0.098344,0.297805,0.000000,0.000000,0.000000,0.000000,1.000000
psychhx,5735.0,0.067306,0.250573,0.000000,0.000000,0.000000,0.000000,1.000000
chrpulhx,5735.0,0.189887,0.392246,0.000000,0.000000,0.000000,0.000000,1.000000
